In [ ]:
pip install tensorflow pandas matplotlib seaborn opencv-python scikit-learn

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("danushkumarv/indian-monuments-image-dataset")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data and visualize folder structures
data_path = os.path.join(path, "Indian-monuments", "images") # Fix: Corrected data_path to point to the 'images' directory
categories = os.listdir(data_path)

print(f"Total Categories: {len(categories)}")
for category in categories:
    print(f"{category}: {len(os.listdir(os.path.join(data_path, category)))} images")

# Bar plot for category distribution
category_counts = [len(os.listdir(os.path.join(data_path, cat))) for cat in categories]
plt.figure(figsize=(12, 6))
sns.barplot(x=categories, y=category_counts)
plt.xticks(rotation=45)
plt.title("Number of Images per Historical Structure")
plt.show()

In [ ]:
import os

# List the contents of the 'Indian-monuments' directory
indian_monuments_path = os.path.join(path, "Indian-monuments")
print(f"Contents of {indian_monuments_path}:")
for item in os.listdir(indian_monuments_path):
    print(item)

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Parameters
IMG_SIZE = 128  # Resize images to 128x128
X_images = []
y_labels = []

# Assuming 'data_path' is defined from previous cells as '/kaggle/input/indian-monuments-image-dataset/Indian-monuments/images'

# First, collect all unique monument names to create labels for classification
all_monument_names = set()
for split_folder in ['test', 'train']: # Iterate through 'test' and 'train' subdirectories
    split_path = os.path.join(data_path, split_folder)
    if os.path.exists(split_path) and os.path.isdir(split_path):
        for monument_name in os.listdir(split_path):
            monument_path = os.path.join(split_path, monument_name)
            if os.path.isdir(monument_path): # Only add if it's a directory (a monument category)
                all_monument_names.add(monument_name)

# Sort the monument names for consistent indexing
unique_monuments = sorted(list(all_monument_names))
num_classes = len(unique_monuments)
monument_to_idx = {name: i for i, name in enumerate(unique_monuments)}

print(f"Found {num_classes} unique monument categories: {unique_monuments}")

# Now, load the images and assign numerical labels
for split_folder in ['test', 'train']:
    split_path = os.path.join(data_path, split_folder)
    if not (os.path.exists(split_path) and os.path.isdir(split_path)):
        print(f"Warning: Split folder not found or not a directory: {split_path}. Skipping.")
        continue

    for monument_name in os.listdir(split_path):
        monument_path = os.path.join(split_path, monument_name)

        if os.path.isdir(monument_path):
            label_idx = monument_to_idx[monument_name]
            for img_file in os.listdir(monument_path):
                img_path = os.path.join(monument_path, img_file)

                # Check if the file is an image file
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img = cv2.imread(img_path)

                    # Check if image was loaded successfully
                    if img is not None:
                        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                        X_images.append(img)
                        y_labels.append(label_idx)
                    else:
                        print(f"Failed to load image: {img_path}")
                else:
                    print(f"Skipping non-image file: {img_path}")
        else:
            print(f"Skipping non-directory item in {split_path}: {monument_name}")

# Convert to numpy arrays and normalize
X = np.array(X_images) / 255.0  # Normalize pixel values to [0, 1]
y = np.array(y_labels)

# Train-test split
if len(X) == 0:
    print("Error: No images were loaded. Cannot perform train-test split.")
else:
    # Use stratify to ensure proportional representation of classes in splits
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)
    print(f"Successfully loaded {len(X)} images.")
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# Load the pre-trained MobileNetV2 model without the top classification layer
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze the layers of the base model
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers on top of the base model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

# Create the new model
transfer_model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
transfer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
transfer_model.summary()

In [ ]:
# Train the transfer learning model
history_transfer = transfer_model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

In [ ]:
# Plot accuracy and loss curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_transfer.history['accuracy'], label='Train Accuracy')
plt.plot(history_transfer.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history_transfer.history['loss'], label='Train Loss')
plt.plot(history_transfer.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss')

plt.show()

# Evaluate the model on test data
test_loss, test_acc = transfer_model.evaluate(X_test, y_test, verbose=2)
print(f"Test Accuracy: {test_acc:.2f}")

In [ ]:
def recommend_structure(image_path):
    # Preprocess uploaded image
    img = cv2.imread(image_path)
    # It's good practice to check if the image was loaded successfully before resizing
    if img is None:
        print(f"Error: Unable to load image at {image_path}")
        return "Error: Image not found or invalid."
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = np.array(img) / 255.0
    img = np.expand_dims(img, axis=0)  # Add batch dimension

    # Predict category
    predictions = transfer_model.predict(img) # Fix: Use the correct model variable 'transfer_model'
    predicted_class = unique_monuments[np.argmax(predictions)] # Fix: Use the correct class names variable 'unique_monuments'

    print(f"Predicted Structure: {predicted_class}")

    # Example recommendations (can be more complex based on marketing data)
    recommendations = {
        "Taj_Mahal": "Promote heritage tours to Agra.",
        "Eiffel_Tower": "Offer exclusive Paris travel packages.",
        "Great_Wall_of_China": "Highlight trekking tours in China.",
        "Sun Temple Konark": "Explore the architectural marvels of the Sun Temple in Konark, Odisha!"
    }

    # Add generic recommendations for all other unique monuments if not already present
    for monument in unique_monuments:
        if monument not in recommendations:
            recommendations[monument] = f"Discover the beauty and history of {monument}."

    return recommendations.get(predicted_class, "No recommendation available.")

# Test the recommendation function
# Fix: Corrected the image path to reflect the Kaggle input directory
recommendation = recommend_structure("/kaggle/input/indian-monuments-image-dataset/Indian-monuments/images/test/Sun Temple Konark/12.jpg")
print(f"Recommendation: {recommendation}")
